# Assign cell types and create pseudobulk

+ Input: 'adata_clusters.h5ad'
+ Output: '{cell_type}_pseudobulk.tsv'
+ Next step: TMM normalisation and Covariate generation in R (perhaps incorporate here later using rpy2)

In [ ]:
# This cell is labelled 'parameters' to work with papermill remotely
# Papermill will overwite the default local plate value below with whatever is passed to 
# the -p flag in the snakerule shell script
#os.system("conda activate eqtl_study") use this locally if using VScode
plate = 'plate1'
plate = globals().get("plate")
print(f"Processing plate: {plate}")

In [ ]:
# Import custom utility packages, lists and functions
import sys
import os
if os.path.exists('/scratch/'):
    root_dir = '/scratch/c.c1477909/eQTL_study_2025/'
else:
    root_dir = '/Users/darren/Desktop/eQTL_study_2025/'
        
sys.path.append(root_dir + 'workflow/scripts/')

from init_env import *
from anndata_utils import *
from gene_lists import *

# Set variables
resolutions = [0.1] 

In [ ]:
# Initialize the environment and get all paths and logger
logger, root_dir, sheets_dir, plate_path, scanpy_dir = initialize_env(plate)
logger.info("Loading data ...")
adata = sc.read(scanpy_dir + f'adata_clusters.h5ad')
adata.X = adata.layers["counts"].copy() # May need to use raw here instead of counts
if adata.raw is not None:
    print("adata.raw exists")
else:
    print("adata.raw does not exist")
adata

# Find overlapping samples with test Genotype data

In [85]:
overlap_file = Path(root_dir + 'resources/test_genotypes/overlapping_samples.txt')

if not overlap_file.exists():
    # Load VCF sample IDs
    with open("/Users/darren/Desktop/eQTL_study_2025/resources/test_genotypes/vcf_samples.txt", "r") as f:
        vcf_samples = set(f.read().splitlines())
    vcf_samples
    # Load snRNA-seq sample IDs (modify based on your format)
    snrna_samples = set(adata.obs["sample"].unique())  # Adjust column name if needed
    snrna_samples

    # # Find overlapping samples
    overlapping_samples = vcf_samples.intersection(snrna_samples)
    print(f"Overlapping samples: {len(overlapping_samples)} / {len(snrna_samples)}")

    # Save overlapping samples list
    with open(overlap_file, "w") as f:
        f.write("\n".join(overlapping_samples))
else:
    print("Overlapping samples file already exists")
    with open(overlap_file, "r") as f:
        overlapping_samples = set(f.read().splitlines())


Overlapping samples file already exists


In [86]:
len(overlapping_samples)

34

# Set cluster IDs

In [ ]:
# Set cluster names
cluster_anns = {
    
    '0': 'cell_type_1',
    '1': 'cell_type_2',
    '2': 'cell_type_3',
    '3': 'cell_type_4',
    '4': 'cell_type_5',
    '5': 'cell_type_6',
    '6': 'cell_type_7',
    '7': 'cell_type_8',
}

# Create a new column in adata.obs with cell type names
adata.obs['cell_type'] = adata.obs['leiden_harmony_0.1'].map(cluster_anns)

# Check the new column
print(adata.obs[['leiden_harmony_0.1', 'cell_type']].head())

In [ ]:
adata.obs['cell_type'].value_counts()

# Manually generate Pseudobulk

+ I had issues trying to generate pseudobulk with [decoupler](https://github.com/saezlab/decoupler-py/issues/166) so left that method for now

In [ ]:
pseudblk_dfs = {}

for cell_type in adata.obs.cell_type.unique():
    print(f"Processing cell type: {cell_type}")
    
    # Subset the data for the current cell type (create a copy)
    cell_subset = adata[adata.obs['cell_type'] == cell_type].copy()
    
    # Initialize an empty list to store pseudobulk objects
    combined_pseudobulk_lst = []

    # Loop through unique samples
    for sample in cell_subset.obs['sample'].unique():

        # Remove samples not in the overlapping set
        if sample not in overlapping_samples:  
            continue

        samp_cell_subset = cell_subset[cell_subset.obs['sample'] == sample].copy()

        # Extract the raw counts
        counts_matrix = samp_cell_subset.layers["counts"]
        
        # Ensure counts are stored as integers
        print(f"Max count value (should be integer): {counts_matrix.max()}")
        
        # Sum counts across cells in the sample
        pseudobulk_counts = counts_matrix.sum(axis=0).A1 if hasattr(counts_matrix, "A1") else counts_matrix.sum(axis=0)
        
        # Create an AnnData object with summed counts
        samp_adata = sc.AnnData(X=pseudobulk_counts.reshape(1, -1), var=samp_cell_subset.var)
        samp_adata.obs_names = [sample]

        # Append to the list
        combined_pseudobulk_lst.append(samp_adata)

    # Combine all samples into one AnnData object
    pseudobulk_adata = sc.concat(combined_pseudobulk_lst, join="outer")

    # Convert pseudobulk_adata to a DataFrame for the current cell type
    pseudobulk_df = pd.DataFrame(pseudobulk_adata.X, columns=pseudobulk_adata.var_names, index=pseudobulk_adata.obs_names)
    print(f"pseudobulk_df dims: {pseudobulk_df.shape}")

    # Save the DataFrame to the dictionary
    pseudblk_dfs[cell_type] = pseudobulk_df

    # Save the TMM-corrected pseudobulk data for this cell type to a CSV file
    pseudobulk_df.to_csv(os.path.join(scanpy_dir, f'pseudobulk/{cell_type}_pseudobulk.csv'))

In [ ]:
for df in pseudblk_dfs:
    print(pseudblk_dfs[df].shape)

In [ ]:
# Determining Covariates : Need to do this in R
# Extract PC loadings
# pc_df = pd.DataFrame(adata.obsm["X_pca"], index=adata.obs.index)

# # Check correlation with potential covariates
# covariates = ["sublibrary", "plate"]  # Modify based on your data
# corr_df = pc_df.join(adata.obs[covariates])
# corr_df = pd.get_dummies(corr_df, columns=["plate"], drop_first=True)
# correlations = corr_df.corr()
# print(correlations)

# plt.figure(figsize=(10, 8))

# # Generate heatmap
# sns.heatmap(correlations, cmap="coolwarm", linewidths=0.5, vmin=-1, vmax=1)
# plt.title("Correlation Matrix Heatmap")
# plt.show()

